# Week 5 - Apache Spark Data Processing Assignment
### Name: Eklavya
### Internship: Celebal Technologies Data Engineering Internship
### Week: 5

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Week5SparkAssignment") \
    .getOrCreate()

print("Spark Started Successfully")

Spark Started Successfully


In [3]:
df = spark.read.csv(
    r"Sample - Superstore.csv",
    header=True,
    inferSchema=True,
    multiLine=True,
    quote='"',
    escape='"'
)

In [11]:
print("Rows:", df.count())
print("Columns:", len(df.columns))

df.show(5)

Rows: 9994
Columns: 21
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   B

In [12]:
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



In [13]:
df = df.withColumn(
    "Sales",
    regexp_replace(col("Sales"), ",", "").cast("double")
)

df = df.withColumn(
    "Quantity",
    col("Quantity").cast("int")
)

df = df.withColumn(
    "Discount",
    col("Discount").cast("double")
)

## Q1: What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

Traditional MapReduce has the following limitations:

1. Disk-based processing causes slower execution.
2. High latency due to repeated read/write operations.
3. Complex programming model.
4. Poor support for iterative algorithms.
5. Not suitable for real-time analytics.

Spark overcomes these limitations through in-memory computation and faster execution.

## Q2: Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

Spark stores intermediate results in memory (RAM) instead of writing them repeatedly to disk. This reduces I/O operations and significantly improves performance for iterative machine learning algorithms.

## Q3: Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns.

In [14]:
df_no_duplicates = df.dropDuplicates(["Order ID"])

print("Original Count:", df.count())
print("After Removing Duplicates:", df_no_duplicates.count())

Original Count: 9994
After Removing Duplicates: 5009


## Q4: Filter rows where Region is 'West' and find the average Sales for each Category.

In [15]:
df.filter(col("Region") == "West") \
  .groupBy("Category") \
  .agg(avg("Sales").alias("Average_Sales")) \
  .show()

+---------------+----------------+
|       Category|   Average_Sales|
+---------------+----------------+
|Office Supplies|116.422376910912|
|      Furniture|357.302324611033|
|     Technology|420.687532554257|
+---------------+----------------+



## Q5: Difference between .na.drop() and .na.fill()

- .na.drop() removes rows containing null values.
- .na.fill() replaces null values with specified values.

In [16]:
df.na.fill({
    "City": "Unknown"
}).show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

## Q6: Find the total count of records for each city where the count is greater than 100.

In [17]:
df.groupBy("City") \
  .agg(count("*").alias("Total_Records")) \
  .filter(col("Total_Records") > 100) \
  .show()

+-------------+-------------+
|         City|Total_Records|
+-------------+-------------+
|  Springfield|          163|
|       Dallas|          157|
| Philadelphia|          537|
|  Los Angeles|          747|
|San Francisco|          510|
|    San Diego|          170|
|      Detroit|          115|
|     Columbus|          222|
|      Chicago|          314|
|      Seattle|          428|
|New York City|          915|
|      Houston|          377|
| Jacksonville|          125|
+-------------+-------------+



## Q7: How does DataFrame immutability affect data cleaning?

Spark DataFrames are immutable. Operations such as filtering, dropping columns, and renaming columns create new DataFrames instead of modifying the original DataFrame.

## Q8: Filter rows where age is between 18 and 30 and subscription is Premium.

The Superstore dataset does not contain Age or Subscription columns. Therefore, an equivalent filter is applied using Region and Category.

In [18]:
df.filter(
    (col("Region") == "West") &
    (col("Category") == "Furniture")
).show()

+------+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+----------------+----------+-----------+------+---------------+---------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|   Customer Name|    Segment|      Country|            City|     State|Postal Code|Region|     Product ID| Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+----------------+----------+-----------+------+---------------+---------+------------+--------------------+--------+--------+--------+--------+
|     6|CA-2014-115812|  6/9/2014| 6/14/2014|Standard Class|   BH-11710| Brosina Hoffman|   Consumer|United States|     Los Angeles|California|      90032|  West|FUR-FU-10001487|Furniture| Furnishings|Eldon Expressions...| 

## Q9: Why should null values be handled before aggregation?

Handling null values before aggregation ensures accurate calculations and prevents incorrect analytical results.


## Q10: Convert Order Date into a proper date format.

In [20]:
df = df.withColumn(
    "Order_Date_New",
    to_date(col("Order Date"), "M/d/yyyy")
)

df.select("Order Date", "Order_Date_New").show(5)

+----------+--------------+
|Order Date|Order_Date_New|
+----------+--------------+
| 11/8/2016|    2016-11-08|
| 11/8/2016|    2016-11-08|
| 6/12/2016|    2016-06-12|
|10/11/2015|    2015-10-11|
|10/11/2015|    2015-10-11|
+----------+--------------+
only showing top 5 rows


## Q11: Explain Shuffle Operation.

Shuffle occurs when Spark redistributes data between partitions during operations such as groupBy and join. It is considered a wide transformation because data moves across partitions.

## Q12: Remove invalid rows.

The Superstore dataset does not contain Email or Username columns. Therefore, rows with null City values are removed as an equivalent cleaning operation.

In [21]:
clean_df = df.filter(
    col("City").isNotNull()
)

clean_df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+--------------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|Order_Date_New|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+--------------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-1000179

## Q13: Calculate minimum, maximum and average Sales using .agg().

In [22]:
df.agg(
    min("Sales").alias("Min_Sales"),
    max("Sales").alias("Max_Sales"),
    avg("Sales").alias("Avg_Sales")
).show()

+---------+---------+-----------------+
|Min_Sales|Max_Sales|        Avg_Sales|
+---------+---------+-----------------+
|    0.444| 22638.48|229.8580008304938|
+---------+---------+-----------------+



## Q14: What is the risk of using inferSchema=True?

If source data contains inconsistent formats, Spark may infer incorrect data types, causing schema mismatches and processing errors.

## Q15: Build a processing pipeline that:
1. Removes duplicates
2. Fills null values
3. Groups data and calculates total revenue

In [23]:
result = (
    df
    .dropDuplicates()
    .na.fill({"Sales": 0})
    .groupBy("Region")
    .agg(sum("Sales").alias("Total_Revenue"))
)

result.show()

+-------+------------------+
| Region|     Total_Revenue|
+-------+------------------+
|  South|391721.90500000026|
|Central|501239.89079999976|
|   East| 678781.2400000005|
|   West| 725457.8245000017|
+-------+------------------+



# Conclusion

In this assignment, Apache Spark DataFrames were used to perform:

- Data Cleaning
- Handling Null Values
- Removing Duplicates
- Filtering Data
- Aggregations
- Grouping Operations
- Schema Transformations

Spark's in-memory processing provides significant performance improvements over traditional MapReduce systems.